In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
from src.embeddings.embedder import get_model, embed_texts, embed_single, build_article_text

In [27]:
# Load the cleaned articles dataset from Lab 10
df = pd.read_csv('../data/processed/cleaned/articles_clean.csv')
print(f"Loaded {len(df)} articles")

Loaded 297 articles


In [28]:
print(df[['title', 'description', 'source_name']].head(3))

                                               title  \
0  Scientists discover diet that tricks the body ...   
1  Nature’s powerhouses: The top 8 healthiest ber...   
2  6 Things You Should Do After 5 P.M. to Support...   

                                         description      source_name  
0  Researchers found that cutting two amino acids...    Science Daily  
1  Berries are rich in vitamin C, fiber and polyp...  Naturalnews.com  
2  Aging is a privilege—support your health in la...   Eatingwell.com  


In [29]:
# Sample health article descriptions to explore embeddings
sample_texts = [
    "A new study shows that a Mediterranean diet reduces the risk of heart disease",
    "Researchers find that regular exercise lowers blood pressure and improves cardiovascular health",
    "Mental health experts recommend mindfulness meditation to reduce stress and anxiety",
    "Scientists discover a new drug treatment for type 2 diabetes with fewer side effects",
    "Poor sleep quality is linked to increased risk of obesity and metabolic disorders",
]

# Generate embeddings - each text becomes a vector of 384 numbers
embeddings = embed_texts(sample_texts)

print(f"Shape: {embeddings.shape}")  # (5, 384)
print(f"Type:  {type(embeddings)}")
print(f"First 8 numbers of embedding 0: {embeddings[0][:8]}")

Shape: (5, 384)
Type:  <class 'numpy.ndarray'>
First 8 numbers of embedding 0: [ 0.0114909   0.08202933 -0.01312633  0.04240679 -0.00290348  0.05356296
 -0.06998678 -0.04241271]


In [30]:
from sentence_transformers import util
sim_matrix = util.cos_sim(embeddings, embeddings)

print("Cosine Similarity Matrix:")
print("(Rows and columns match the sample_texts list above)")
print()
labels = ["Diet", "Exercise", "Mental health", "Diabetes", "Sleep"]
for i, row_label in enumerate(labels):
    scores = [f"{sim_matrix[i][j].item():.2f}" for j in range(len(labels))]
    print(f"{row_label:15s}: {' | '.join(scores)}")

print()
print("Expected: Diet vs Exercise should score higher than Diet vs Mental health")

Cosine Similarity Matrix:
(Rows and columns match the sample_texts list above)

Diet           : 1.00 | 0.49 | 0.20 | 0.22 | 0.30
Exercise       : 0.49 | 1.00 | 0.34 | 0.27 | 0.32
Mental health  : 0.20 | 0.34 | 1.00 | 0.06 | 0.15
Diabetes       : 0.22 | 0.27 | 0.06 | 1.00 | 0.30
Sleep          : 0.30 | 0.32 | 0.15 | 0.30 | 1.00

Expected: Diet vs Exercise should score higher than Diet vs Mental health


In [31]:
import torch

query = "healthy eating habits to prevent chronic disease"
query_embedding = embed_single(query)
scores = util.cos_sim(query_embedding, embeddings)[0]
top_results = torch.topk(scores, k=len(sample_texts))

print(f"Query: '{query}'")
print()
print("Ranked results:")
for rank, (score, idx) in enumerate(zip(top_results.values, top_results.indices)):
    print(f"  {rank+1}. [{score:.4f}] {sample_texts[idx]}")

Query: 'healthy eating habits to prevent chronic disease'

Ranked results:
  1. [0.4735] A new study shows that a Mediterranean diet reduces the risk of heart disease
  2. [0.4183] Researchers find that regular exercise lowers blood pressure and improves cardiovascular health
  3. [0.3460] Poor sleep quality is linked to increased risk of obesity and metabolic disorders
  4. [0.2220] Mental health experts recommend mindfulness meditation to reduce stress and anxiety
  5. [0.1987] Scientists discover a new drug treatment for type 2 diabetes with fewer side effects


In [32]:
from sklearn.metrics.pairwise import euclidean_distances

text_a = "A new study shows that a Mediterranean diet reduces the risk of heart disease"
text_b = "Researchers find that regular exercise lowers blood pressure and improves cardiovascular health"
text_c = "Mental health experts recommend mindfulness meditation to reduce stress and anxiety"

emb_a = embed_single(text_a, normalize=True)
emb_b = embed_single(text_b, normalize=True)
emb_c = embed_single(text_c, normalize=True)

cos_ab = util.cos_sim(emb_a, emb_b).item()
cos_ac = util.cos_sim(emb_a, emb_c).item()

dot_ab = float(np.dot(emb_a, emb_b))
dot_ac = float(np.dot(emb_a, emb_c))

euc_ab = float(np.linalg.norm(emb_a - emb_b))
euc_ac = float(np.linalg.norm(emb_a - emb_c))

print("Comparing 'Mediterranean diet' (A) with:")
print(f"  B = 'Exercise & cardiovascular': cosine={cos_ab:.4f}, dot={dot_ab:.4f}, euclidean={euc_ab:.4f}")
print(f"  C = 'Mindfulness meditation':    cosine={cos_ac:.4f}, dot={dot_ac:.4f}, euclidean={euc_ac:.4f}")
print()
print("Conclusion: A and B (both physical health) should be more similar than A and C.")

Comparing 'Mediterranean diet' (A) with:
  B = 'Exercise & cardiovascular': cosine=0.4909, dot=0.4909, euclidean=1.0090
  C = 'Mindfulness meditation':    cosine=0.2041, dot=0.2041, euclidean=1.2617

Conclusion: A and B (both physical health) should be more similar than A and C.


In [33]:
from src.embeddings.chroma_store import get_chroma_client, get_collection, add_articles_to_collection

client = get_chroma_client()
collection = get_collection(client=client, reset=False)

print(f"Collection '{collection.name}' ready")
print(f"Current count: {collection.count()} articles")

Collection 'health_articles' ready
Current count: 297 articles


In [34]:
df = pd.read_csv('../data/processed/cleaned/articles_clean.csv')

total = add_articles_to_collection(df, collection, batch_size=100)
print(f"Done. {total} articles are now searchable by meaning.")

Collection already has 297 articles
Collection now contains 297 articles
Done. 297 articles are now searchable by meaning.


In [35]:
results = collection.query(
    query_texts=["diet and nutrition tips for a healthy lifestyle"],
    n_results=3
)

print("Top 3 results for 'diet and nutrition tips for a healthy lifestyle':")
print()
for i, (doc, meta, dist) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
)):
    similarity = 1 - dist
    print(f"{i+1}. [{similarity:.3f}] {meta['title']} ({meta['publish_year']}) - {meta['source_name']}")
    print(f"   {doc[:120]}...")
    print()

Top 3 results for 'diet and nutrition tips for a healthy lifestyle':

1. [0.475] Mediterranean Diet Linked to Lower Risk of Heart Disease (0) - WebMD
   Mediterranean Diet Linked to Lower Risk of Heart Disease | New research confirms that adherence to the Mediterranean die...

2. [0.469] How worried should you be about your BMI? (0) - New Scientist
   How worried should you be about your BMI? | Body mass index (BMI) is used as a global standard for measuring health, but...

3. [0.468] The fiber gap: Why whole foods outshine supplements for optimal health (0) - Naturalnews.com
   The fiber gap: Why whole foods outshine supplements for optimal health | Fiber is a critical yet overlooked health power...



In [36]:
queries = [
    "mental health and stress management",
    "cancer research and new treatments",
    "weight loss and obesity prevention",
]

results = collection.query(
    query_texts=queries,
    n_results=3
)

for q_idx, query in enumerate(queries):
    print(f"Query: '{query}'")
    for rank in range(3):
        meta = results["metadatas"][q_idx][rank]
        dist = results["distances"][q_idx][rank]
        sim  = 1 - dist
        print(f"  {rank+1}. [{sim:.3f}] {meta['title']} ({meta['publish_year']})")
    print()

Query: 'mental health and stress management'
  1. [0.445] Dealing with stress-caused sickness in family caregivers (0)
  2. [0.437] Mental Health and Exercise: What's the Connection? (0)
  3. [0.437] Mental Health and Exercise: What's the Connection? (0)

Query: 'cancer research and new treatments'
  1. [0.495] Your morning coffee could one day help fight cancer (0)
  2. [0.475] The Future Of Cancer Treatment Could Involve Your Morning Coffee (0)
  3. [0.364] The US slashed research for cancer, Alzheimer’s, mental health — and nearly everything else (0)

Query: 'weight loss and obesity prevention'
  1. [0.566] Relying on drugs to stop obesity would be 'societal failure', says Chris Whitty (0)
  2. [0.560] Hunger Games: Why Diets Fail and Weight Loss Medicines Succeed (0)
  3. [0.527] Best science-based weight loss program for long-term results (0)



Filter by Source

In [37]:
top_source = df['source_name'].value_counts().index[0]

results = collection.query(
    query_texts=["diabetes treatment and blood sugar management"],
    where={"source_name": top_source},
    n_results=5
)

print(f"Articles about 'diabetes treatment' from source '{top_source}':")
for meta, dist in zip(results["metadatas"][0], results["distances"][0]):
    print(f"  [{1-dist:.3f}] {meta['title']} ({meta['publish_year']})")

Articles about 'diabetes treatment' from source 'Naturalnews.com':
  [0.561] Moving more could stop hundreds of thousands of diabetes crises before they start (0)
  [0.538] Diabetes-friendly pantry: How shelf-stable foods can stabilize blood sugar (0)
  [0.528] Gymnema blocks sugar cravings, helping diabetic patients (0)
  [0.495] A tangy brew for metabolic health: Kombucha shows promise in blood sugar management (0)
  [0.477] Coconut water: A natural ally in managing blood sugar and supporting overall wellness (0)


Filter by Year Range

In [38]:
# Find recent articles (2020 and later) about mental health
results = collection.query(
    query_texts=["mental health anxiety and depression"],
    where={"publish_year": {"$gte": 2020}},
    n_results=5
)

print("Recent mental health articles (2020+):")
for meta, dist in zip(results["metadatas"][0], results["distances"][0]):
    print(f"  [{1-dist:.3f}] {meta['title']} ({meta['publish_year']}) - {meta['source_name']}")

Recent mental health articles (2020+):


Filter by Multiple Conditions

In [39]:
results = collection.query(
    query_texts=["nutrition advice for heart health"],
    where={
        "$and": [
            {"publish_year": {"$gte": 2020}},
            {"publish_year": {"$lte": 2026}},
        ]
    },
    n_results=5
)

print("Nutrition & heart health articles from 2020-2026:")
for meta, dist in zip(results["metadatas"][0], results["distances"][0]):
    print(f"  [{1-dist:.3f}] {meta['title']} ({meta['publish_year']}, source={meta['source_name']})")

Nutrition & heart health articles from 2020-2026:


In [40]:
from src.embeddings.chroma_store import get_chroma_client, get_collection
from src.embeddings.search_engine import compare_search

In [41]:
df = pd.read_csv('../data/processed/cleaned/articles_clean.csv')

client = get_chroma_client()
collection = get_collection(client)

test_queries = [
    "healthy diet and weight management",
    "stress and mental wellness",
    "cancer prevention and early detection",
    "exercise benefits for the elderly",
]

for q in test_queries:
    results = compare_search(q, df, collection=collection, n_results=3)
    print("=" * 60)

--- Query: 'healthy diet and weight management' ---

Keyword search found 0 results:

Semantic search found 3 results:
  [0.511] Hunger Games: Why Diets Fail and Weight Loss Medicines Succeed (0) - Psychology Today
  [0.497] Best science-based weight loss program for long-term results (0) - TechRadar
  [0.473] How worried should you be about your BMI? (0) - New Scientist
--- Query: 'stress and mental wellness' ---

Keyword search found 0 results:

Semantic search found 3 results:
  [0.537] Parents’ stress may be quietly driving childhood obesity, Yale study finds (0) - Science Daily
  [0.510] Dealing with stress-caused sickness in family caregivers (0) - Scientific American
  [0.483] Mental Health and Exercise: What's the Connection? (0) - Mayo Clinic Staff
--- Query: 'cancer prevention and early detection' ---

Keyword search found 0 results:

Semantic search found 3 results:
  [0.373] The Future Of Cancer Treatment Could Involve Your Morning Coffee (0) - mindbodygreen.com
  [0.372] Y

In [42]:
from src.embeddings.search_engine import keyword_search, semantic_search

In [43]:
import matplotlib.pyplot as plt

test_cases = [
    {"query": "heart disease prevention",  "synonym": "cardiovascular health tips"},
    {"query": "mental health treatment",   "synonym": "therapy for anxiety and depression"},
    {"query": "diabetes management",       "synonym": "blood sugar control strategies"},
]

for case in test_cases:
    q = case["query"]
    q2 = case["synonym"]

    kw1 = keyword_search(q,  df, n_results=5)
    kw2 = keyword_search(q2, df, n_results=5)
    sem1 = semantic_search(q,  n_results=5, collection=collection)
    sem2 = semantic_search(q2, n_results=5, collection=collection)

    titles_kw1  = set(kw1['title'].tolist())
    titles_kw2  = set(kw2['title'].tolist())
    titles_sem1 = set(sem1['title'].tolist())
    titles_sem2 = set(sem2['title'].tolist())

    kw_overlap  = len(titles_kw1  & titles_kw2)
    sem_overlap = len(titles_sem1 & titles_sem2)

    print(f"Query pair: '{q}' vs '{q2}'")
    print(f"  Keyword overlap:  {kw_overlap}/5 articles in common")
    print(f"  Semantic overlap: {sem_overlap}/5 articles in common")
    print("  (Higher overlap = more consistent results across synonyms)")
    print()

Query pair: 'heart disease prevention' vs 'cardiovascular health tips'
  Keyword overlap:  0/5 articles in common
  Semantic overlap: 3/5 articles in common
  (Higher overlap = more consistent results across synonyms)

Query pair: 'mental health treatment' vs 'therapy for anxiety and depression'
  Keyword overlap:  0/5 articles in common
  Semantic overlap: 3/5 articles in common
  (Higher overlap = more consistent results across synonyms)

Query pair: 'diabetes management' vs 'blood sugar control strategies'
  Keyword overlap:  0/5 articles in common
  Semantic overlap: 0/5 articles in common
  (Higher overlap = more consistent results across synonyms)



In [44]:
from src.embeddings.hybrid_search import hybrid_search

In [45]:
query = "tips for staying healthy and preventing illness"
n = 5

# Individual results
kw  = keyword_search(query, df, n_results=n)
sem = semantic_search(query, collection=collection, n_results=n)
hyb = hybrid_search(query, df, collection, n_results=n)

print(f"KEYWORD SEARCH ({len(kw)} results):")
for _, row in kw.iterrows():
    print(f"  {row['title']} ({row.get('publish_year', '?')})")

print()
print(f"SEMANTIC SEARCH ({len(sem)} results):")
for _, row in sem.iterrows():
    print(f"  [{row['similarity']:.3f}] {row['title']} ({row['publish_year']})")

print()
print(f"HYBRID SEARCH / RRF ({len(hyb)} results):")
for _, row in hyb.iterrows():
    print(f"  [rrf={row['rrf_score']:.4f}] {row['title']} ({row.get('publish_year', '?')})")

KEYWORD SEARCH (0 results):

SEMANTIC SEARCH (5 results):
  [0.448] Dealing with stress-caused sickness in family caregivers (0)
  [0.426] This Health Factor Increases Heart Disease Risk By Almost 20%, Study Shows (0)
  [0.395] Eating This Way Could Lower Risk Of Gastric Cancer By Almost 30%, Study Finds (0)
  [0.394] 6 Things You Should Do After 5 P.M. to Support Healthy Aging, According to Experts (0)
  [0.384] Natural Light May Help Manage This Common Disease, Research Shows (0)

HYBRID SEARCH / RRF (5 results):
  [rrf=0.0164] Dealing with stress-caused sickness in family caregivers (0)
  [rrf=0.0161] This Health Factor Increases Heart Disease Risk By Almost 20%, Study Shows (0)
  [rrf=0.0159] Eating This Way Could Lower Risk Of Gastric Cancer By Almost 30%, Study Finds (0)
  [rrf=0.0156] 6 Things You Should Do After 5 P.M. to Support Healthy Aging, According to Experts (0)
  [rrf=0.0154] Natural Light May Help Manage This Common Disease, Research Shows (0)


Part 13 - Analytical Questions and Examples

Question 1: Which articles are most similar to a specific article?

In [46]:
sample_article = df.iloc[0]
article_text = build_article_text(sample_article)

print(f"Finding articles similar to: {sample_article['title']}")
print(f"Text used: {article_text[:150]}...")
print()

results = semantic_search(article_text, n_results=6, collection=collection)

print("Most similar articles:")
for _, row in results.iloc[1:].iterrows():
    print(f"  [{row['similarity']:.3f}] {row['title']} ({row['publish_year']}) - {row['source_name']}")

Finding articles similar to: Scientists discover diet that tricks the body into burning fat without exercise
Text used: Scientists discover diet that tricks the body into burning fat without exercise | Researchers found that cutting two amino acids common in animal prot...

Most similar articles:
  [0.505] Keto diet could improve response to exercise in people with high blood sugar (0) - Futurity: Research News
  [0.410] 7 Science-Backed Benefits of Intermittent Fasting (0) - Healthline
  [0.401] Hunger Games: Why Diets Fail and Weight Loss Medicines Succeed (0) - Psychology Today
  [0.388] Fathers’ tobacco use linked to metabolic changes in their children (0) - Science Daily
  [0.380] This Workout Is Proven To Boost The Efficiency Of Your Mitochondria (0) - mindbodygreen.com


Question 2: Which sources have the most semantically diverse articles?

In [47]:
from src.embeddings.embedder import embed_texts, build_article_text

sample = df.sample(n=min(500, len(df)), random_state=42).copy()
sample['text'] = sample.apply(build_article_text, axis=1)

print("Generating embeddings for sample articles...")
embs = embed_texts(sample['text'].tolist())
sample['embedding'] = list(embs)

source_diversity = {}
for source, group in sample.groupby('source_name'):
    if len(group) < 3:
        continue
    emb_matrix = np.vstack(group['embedding'].values)
    # Average variance across all embedding dimensions
    diversity = float(np.mean(np.var(emb_matrix, axis=0)))
    source_diversity[source] = round(diversity, 6)

top_sources = sorted(source_diversity.items(), key=lambda x: -x[1])[:10]
print("Most semantically diverse sources:")
for source, score in top_sources:
    print(f"  {source:40s}: diversity = {score:.6f}")

Generating embeddings for sample articles...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Most semantically diverse sources:
  Science Daily                           : diversity = 0.001955
  Scientific American                     : diversity = 0.001938
  Nature.com                              : diversity = 0.001898
  mindbodygreen.com                       : diversity = 0.001819
  Die Zeit                                : diversity = 0.001785
  New Scientist                           : diversity = 0.001752
  Naturalnews.com                         : diversity = 0.001740
  The Indian Express                      : diversity = 0.001705
  tagesschau.de                           : diversity = 0.001679
  Business Insider                        : diversity = 0.001601


Question 3: How do search results change with filters?

In [48]:
def show_results(label, results_df):
    cols = [c for c in ['title', 'publish_year', 'source_name', 'similarity'] if c in results_df.columns]
    if results_df.empty or not cols:
        print(f"{label}: no results")
    else:
        print(f"{label}:")
        print(results_df[cols].to_string(index=False))
    print()

query = "exercise and physical fitness"
top_source = df['source_name'].value_counts().index[0]

unfiltered      = semantic_search(query, n_results=5, collection=collection)
source_filtered = semantic_search(query, n_results=5, collection=collection,
                                  filters={"source_name": top_source})
recent_only     = semantic_search(query, n_results=5, collection=collection,
                                  filters={"publish_year": {"$gte": 2022}})

show_results("No filter", unfiltered)
show_results(f"Top source only ({top_source})", source_filtered)
show_results("2022 and later only", recent_only)

No filter:
                                                                            title  publish_year       source_name  similarity
                               Mental Health and Exercise: What's the Connection?             0 Mayo Clinic Staff      0.5083
                               Mental Health and Exercise: What's the Connection?             0       Mayo Clinic      0.5083
Moving more could stop hundreds of thousands of diabetes crises before they start             0   Naturalnews.com      0.4404
              This Workout Is Proven To Boost The Efficiency Of Your Mitochondria             0 mindbodygreen.com      0.4029
                                        How worried should you be about your BMI?             0     New Scientist      0.3740

Top source only (Naturalnews.com):
                                                                               title  publish_year     source_name  similarity
   Moving more could stop hundreds of thousands of diabetes crises bef